In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../data/clinical/data_June_2025.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2140 entries, 0 to 2139
Columns: 4431 entries, PTID to AB_SUVR_GM_GLOBL
dtypes: float64(4203), int64(69), object(159)
memory usage: 72.3+ MB


/tmp/ipykernel_1443683/3661735777.py:1: DtypeWarning: Columns (0,20,22,34,36,38,42,48,52,66,68,70,75,86,94,603,612,630,631,632,633,634,635,636,637,638,639,640,641,642,643,644,731,740,751,754,764,796,978,982,984,1006,1008,1017,1028,1030,1077,1247,1283,1299,1316,1388,1399,1432,1441,1609,1611,1613,1615,1617,1621,1625,1629,1641,1645,1988,1990,1992,1994,2184,2186,2372,2374,2393,2394,2396,2408,2409,2410,2440,2445,2450,2454,2519,2520,2661,2662,2663,2694,2705,3076,3100,3314,3316,3324,3326,3404,3407,3419,3434,3437,3449,3452,3456,3458,3594,3596,3601,3642,3674,4156,4158) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/clinical/data_June_2025.csv')


# 1.Pre-processing

In [3]:
df = df[~df['PTID'].str.startswith('A', na=False)]

## MMSE Processing
There are two MMSE columns and I would noticed that some of 'FL_MMSE' had a value of 999,-4, or empty so i just use the column 'MMSE' as a fall back to fill them.

Filling out MMSE Values. Using FL_MMSE Column as is the most complete, if the value is 999 or -4 set it to NaN. 
fallback to MMSE values if is missing.

In [4]:
df["COMBINED_MMSE"] = df["FL_MMSE"].replace([999, -4], np.nan)
df["COMBINED_MMSE"] = df["COMBINED_MMSE"].fillna(df["MMSE"])
df = df.drop(columns=["FL_MMSE", "MMSE"])

In [5]:
df = df.rename(columns={"COMBINED_NE4S": "APOE4S", "COMBINED_MMSE": "MMSE"})

In [ ]:
# Demographics & Clinical Identifiers
DEMOGRAPHICS = [
    "PTID",        # Patient ID
    "VISITYR",     # Visit Year
]

# Cognitive & Clinical Assessments
COGNITIVE = [
    "CDRSUM",      # CDR Sum of Boxes
    "MMSE",        # Mini-Mental State Exam
    "HVLT_DR",     # Hopkins Verbal Learning Test - Delayed Recall
    "LASSI_A_CR2", # LASSI List A - Correct Recall 2
    "LASSI_B_CR1", # LASSI List B - Correct Recall 1
    "LASSI_B_CR2", # LASSI List B - Correct Recall 2
]

# Blood / CSF Biomarkers
BIOMARKER = [
    "APOE4S",              # APOE ε4 Status
    "PTAU_217_CONCNTRTN",  # Phospho-Tau 217 Concentration
]

# Brain volumes
BRAIN_VOLUME = [
    "VOL_ENTRHNA_L",
    "VOL_ENTRHNA_R",
    "VOL_HIPP_L",
    "VOL_HIPP_R",
    "VOL_AMYG_L",
    "VOL_AMYG_R",
    # "VOL_PRECUNE_L",
    # "VOL_PRECUNE_R",
    "VOL_POST_CING_L",
    "VOL_POST_CING_R",
    # "VOL_INF_PAR_L",
    # "VOL_INF_PAR_R",
    "VOL_INF_TEMP_L",
    "VOL_INF_TEMP_R",
    "VOL_TEMP_PL_L",
    "VOL_TEMP_PL_R",
    # "VOL_LAT_ORB_L",
    # "VOL_LAT_ORB_R",
    "VOL_SUP_FRNT_L",
    "VOL_SUP_FRNT_R",
    # "VOL_PRECENT_L",
    # "VOL_PRECENT_R",
    # "VOL_HIPP_SBCLM_HEAD_L",
    # "VOL_HIPP_SBCLM_HEAD_R",
    # "VOL_HIPP_SBCLM_BOD_L",
    # "VOL_HIPP_SBCLM_BOD_R",
    # "VOL_HIPP_PRESBCLM_HEAD_L",
    # "VOL_HIPP_PRESBCLM_HEAD_R",
    # "VOL_HIPP_PRESBCLM_BOD_L",
    # "VOL_HIPP_PRESBCLM_BOD_R",
    "VOL_HIPP_CA1_HEAD_L",
    "VOL_HIPP_CA1_HEAD_R",
    "VOL_HIPP_CA1_BOD_L",
    "VOL_HIPP_CA1_BOD_R",
    "VOL_ETIV"
]

BRAIN_THICKNESS =   [
    "THK_ENTRHNA_L",
    "THK_ENTRHNA_R",
    # "THK_PARAHIPP_L",
    # "THK_PARAHIPP_R",
    "THK_PRECUNE_L",
    "THK_PRECUNE_R",
    "THK_POST_CING_L",
    "THK_POST_CING_R",
    # "THK_INF_PAR_L",
    # "THK_INF_PAR_R",
    "THK_INF_TEMP_L",
    "THK_INF_TEMP_R",
    "THK_TEMP_PL_L",
    "THK_TEMP_PL_R",
    # "THK_LAT_ORB_L",
    # "THK_LAT_ORB_R",
    # "THK_ROST_MIDFRNT_L",
    # "THK_ROST_MIDFRNT_R",
    # "THK_CAUD_MIDFRNT_L",
    # "THK_CAUD_MIDFRNT_R",
    "THK_SUP_FRNT_L",
    "THK_SUP_FRNT_R",
    # "THK_PRECENT_L",
    # "THK_PRECENT_R"
]

TARGETS = [
    "NACCETPR",    # Etiology
    "FL_UDSD",     # Cognitive Status
]

In [7]:
FEATURES = DEMOGRAPHICS + COGNITIVE + BIOMARKER + BRAIN_VOLUME + BRAIN_THICKNESS + TARGETS

In [8]:
df_filter = df[FEATURES].copy()

In [9]:
df_filter[BRAIN_VOLUME] = df_filter[BRAIN_VOLUME].div(df_filter["VOL_ETIV"], axis=0)

In [10]:
X_FEATURES = DEMOGRAPHICS + COGNITIVE + BIOMARKER + BRAIN_VOLUME + BRAIN_THICKNESS

In [11]:
df_filter = df_filter.dropna(subset=X_FEATURES)

In [12]:
df_filter.info()

<class 'pandas.core.frame.DataFrame'>
Index: 394 entries, 6 to 2072
Data columns (total 43 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   PTID                 394 non-null    object 
 1   VISITYR              394 non-null    int64  
 2   CDRSUM               394 non-null    float64
 3   MMSE                 394 non-null    float64
 4   HVLT_DR              394 non-null    float64
 5   LASSI_A_CR2          394 non-null    float64
 6   LASSI_B_CR1          394 non-null    float64
 7   LASSI_B_CR2          394 non-null    float64
 8   APOE4S               394 non-null    float64
 9   PTAU_217_CONCNTRTN   394 non-null    float64
 10  VOL_ENTRHNA_L        394 non-null    float64
 11  VOL_ENTRHNA_R        394 non-null    float64
 12  VOL_HIPP_L           394 non-null    float64
 13  VOL_HIPP_R           394 non-null    float64
 14  VOL_AMYG_L           394 non-null    float64
 15  VOL_AMYG_R           394 non-null    float64

In [13]:
df_filter.drop(columns=["VOL_ETIV"], inplace=True)

In [14]:
# WENT FROM 591 ROWS TO 394

In [15]:
# drop_df_filter = df_filter.dropna(subset=["CDRGLOB", "AMYLPET","FL_UDSD", "NACCETPR"])

In [16]:
# drop_df_filter.info()

In [17]:
df_filter.to_csv('../data/clinical/clinical_preprocessed.csv', index=False)